In [112]:
from scipy.optimize import minimize
import numpy as np

In [113]:
# 主題：購買蘋果、香蕉限制金額下效用最大化
# 程式碼示範：以scipy.optimize.minimize進行優化
# 1. 定義目標函數 (Scipy 預設是「最小化」，所以最大化問題要加負號)
def f(x):
    return -x[0]*x[1]
# 2. 定義約束條件
# Scipy 的 'ineq' 代表 g(x) >= 0，所以我們要寫成 100 - 10x - 5y >= 0
def g(x):
    return 100- 10*x[0] - 5*x[1]
con1 = {'type': 'ineq', 'fun': g}
cons=[con1]

# 3. 定義變數的邊界 (x, y 必須大於 0)
bnds = ((0.001, None), (0.001, None))
# 4. 給予初始猜測值
x = np.array([0,0])
# 5. 執行優化
res=minimize(fun=f,x0=x,method='SLSQP',bounds=bnds,constraints=cons)
# 6. 輸出結果
print(f"最佳蘋果購買數 x: {res.x[0]:.2f}")
print(f"最佳香蕉購買數 y: {res.x[1]:.2f}")
print(f"最大效益: {-res.fun:.2f}")
print(f"拉格朗日乘數:{float(res['multipliers'][0]):.2f}")

最佳蘋果購買數 x: 5.00
最佳香蕉購買數 y: 10.00
最大效益: 50.00
拉格朗日乘數:1.00


In [114]:
# 程式碼示範：自訂類別， 以sympy.solvers.solve進行優化
from sympy import symbols, diff
from sympy.solvers import solve
# 自訂類別
class minimizer:
    def __init__(self,constraint):
        self.res_=None
        self.constraint_=constraint
    # 1. 定義目標函數 (預設是「最小化」，所以最大化問題要加負號)
    def f(self,x):
        return -x[0]*x[1]
    # 2. 定義限制條件
    def g(self,x):
        return (10*x[0] + 5*x[1])-self.constraint_
    # 3. 執行優化
    def optimize(self):
        x, y, lambda0 = symbols('x y lambda0', real=True)
        self.x_=x
        self.y_=y
        self.lambda0_=lambda0
        grad=[]
        gradF = [diff(self.f([x,y]), x), diff(self.f([x, y]), y), self.g([x, y])]
        gradG = [diff(self.g([x,y]), x), diff(self.g([x, y]), y)]
        for i in range(0, len(gradF)):
            if(i != len(gradF) - 1):
                grad.append(gradF[i] + lambda0 * gradG[i])
            else:
                grad.append(gradF[i])
        self.res_ = solve(grad, x, y, lambda0)
        return self
    # 4. 顯示結果
    def show_result(self):
        x_star=self.res_[self.x_]
        y_star=self.res_[self.y_]
        print(f"最佳蘋果購買數 x: {self.res_[self.x_]:.2f}")
        print(f"最佳香蕉購買數 y: {self.res_[self.y_]:.2f}")
        print(f"最大效益: {-self.f([x_star,y_star]):.2f}")
        print(f"拉格朗日乘數:{self.res_[self.lambda0_]:.2f}")

# 進行優化
mm=minimizer(constraint=100)
sol=mm.optimize()
# 輸出結果
sol.show_result()

最佳蘋果購買數 x: 5.00
最佳香蕉購買數 y: 10.00
最大效益: 50.00
拉格朗日乘數:1.00


In [115]:
# 與放寬約束後的比較：
# 進行優化
mm=minimizer(constraint=110)
sol=mm.optimize()
# 輸出結果
sol.show_result()

最佳蘋果購買數 x: 5.50
最佳香蕉購買數 y: 11.00
最大效益: 60.50
拉格朗日乘數:1.10


In [116]:
# 成本最小化範例
# 1. 定義目標函數 (Scipy 預設是「最小化」，所以最大化問題要加負號)
def f1(x):    # 目標成本函數
    cost = 20*x[0]**2+40*x[0]*x[1]
    return cost
# 2. 定義約束條件
# Scipy 的 'ineq' 代表 g(x) >= 0，所以我們要寫成 x**2*h - 16 >= 0
def g1(x): # 容量限制
    return x[0]**2*x[1] - 16


In [ ]:
# 3. 定義變數的邊界 (x, y 必須大於 0)
bnds = [(0.001, None),(0.001, None)]    # x、h 都必須大於0
# 4. 給予初始猜測值
x1 = np.array([1,1])
con1 = {'type': 'ineq', 'fun': g1}
cons=[con1]
# 5. 執行優化(最小化)
res=minimize(fun=f1,x0=x1,method='SLSQP',bounds=bnds,constraints=cons)
# 6. 輸出結果
print(res)
print(f"最佳底邊長 x: {res.x[0]:.2f}")
print(f"最佳高度 y: {res.x[1]:.2f}")
print(f"最小成本: {res.fun:.2f}")
print(f"拉格朗日乘數:{float(res['multipliers'][0]):.2f}")

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: 380.9762426885793
           x: [ 2.520e+00  2.520e+00]
         nit: 14
         jac: [ 2.016e+02  1.008e+02]
        nfev: 53
        njev: 13
 multipliers: [ 1.587e+01]
最佳底邊長 x: 2.52
最佳高度 y: 2.52
最小成本: 380.98
拉格朗日乘數:15.87


In [129]:
# 1. 定義目標函數 (預設是「最小化」，所以最大化問題要加負號)
def f2(x):
    return -10*x[0]**0.5*x[1]**0.5
# 2. 定義限制條件
def g2(x):
    return 100-2*x[0] - 5*x[1]

In [131]:
# 3. 定義變數的邊界 (x, y 必須大於 0)
bnds = [(0.001, None),(0.001, None)]    # x、h 都必須大於0
# 4. 給予初始猜測值
x2 = np.array([5,5])
con1 = {'type': 'ineq', 'fun': g2}
cons=[con1]
# 5. 執行優化(最小化)
res=minimize(fun=f2,x0=x2,method='SLSQP',bounds=bnds,constraints=cons)
# 6. 輸出結果
print(res)
print(f"最佳勞動力投入 x: {res.x[0]:.2f}")
print(f"最佳機器投入 y: {res.x[1]:.2f}")
print(f"最大產出量: {-res.fun:.2f}")
print(f"拉格朗日乘數:{float(res['multipliers'][0]):.2f}")

     message: Optimization terminated successfully
     success: True
      status: 0
         fun: -158.1138830085842
           x: [ 2.500e+01  1.000e+01]
         nit: 7
         jac: [-3.162e+00 -7.906e+00]
        nfev: 21
        njev: 7
 multipliers: [ 1.581e+00]
最佳勞動力投入 x: 25.00
最佳機器投入 y: 10.00
最大產出量: 158.11
拉格朗日乘數:1.58
